# Notebook 05 — NLP Fine-Tuning
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook fine-tunes five pre-trained transformer models on the natural-language student narratives produced by Notebook 01. Each model learns to classify a student's mental health status (`Stable`, `Challenged`, `Critical`) from a plain-English description of their demographics and all 26 psychometric item responses.

---

## Models

| Key | HuggingFace ID | Rationale |
|-----|---------------|----------|
| `bert` | `bert-base-uncased` | General-purpose transformer baseline |
| `distilbert` | `distilbert-base-uncased` | Lightweight distilled BERT — faster on CPU |
| `albert` | `albert-base-v2` | Parameter-efficient; shares weights across layers |
| `biobert` | `dmis-lab/biobert-v1.1` | Pre-trained on PubMed + PMC biomedical text |
| `clinicalbert` | `emilyalsentzer/Bio_ClinicalBERT` | Pre-trained on MIMIC-III clinical notes |

BioBERT and ClinicalBERT are included because the questionnaire items map closely to clinical depression/anxiety/stress assessment language.

---

## Training Configuration

| Parameter | Value | Notes |
|-----------|-------|-------|
| `num_train_epochs` | 5 | |
| `per_device_train_batch_size` | 4 | Kept small to stay within 16 GB RAM on CPU |
| `per_device_eval_batch_size` | 8 | |
| `learning_rate` | 2e-5 | Standard for fine-tuning BERT-family models |
| `warmup_ratio` | 0.1 | 10% of steps used for LR warm-up |
| `weight_decay` | 0.01 | L2 regularisation |
| `max_length` | 128 tokens | Sufficient for the narrative length |
| `evaluation_strategy` | epoch | Evaluate after every epoch |
| `metric_for_best_model` | `f1` (macro F1) | Consistent with NB03 and NB04 |
| `load_best_model_at_end` | True | Restores best checkpoint before final eval |
| `no_cuda` | True | CPU-only training |
| `dataloader_num_workers` | 0 | Avoids multiprocessing issues on Windows |

---

## Primary Metric

**Macro F1** — same as NB03 and NB04. The `compute_metrics` function returns `f1` (macro F1), `f1_weighted`, and `accuracy`. The Trainer uses `f1` for checkpoint selection and early stopping.

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports and working directory fix |
| 2 | Load text dataset · encode labels · stratified split · `compute_metrics` · `fine_tune_transformer` helper |
| 3 | Fine-tune BERT |
| 4 | Fine-tune DistilBERT |
| 5 | Fine-tune ALBERT |
| 6 | Fine-tune BioBERT |
| 7 | Fine-tune ClinicalBERT |
| 8 | Consolidate results · save NLP summary CSV · print sorted by F1_Macro |

## Cell 1 — Imports and Working Directory

Imports all libraries. HuggingFace `transformers` and `datasets` are used for model loading, tokenisation, and training. `sentencepiece` is required by ALBERT. The `tokenizers` parallelism warning is suppressed since `dataloader_num_workers=0` is used throughout.

In [1]:
from pathlib import Path
import os, warnings, shutil
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics         import f1_score, accuracy_score
from datasets                import Dataset
from transformers            import (AutoTokenizer,
                                     AutoModelForSequenceClassification,
                                     Trainer, TrainingArguments)

plt.rcParams.update({'savefig.dpi': 300,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

SEED       = 42
MAX_LENGTH = 128
LABEL_MAP  = {'Stable': 0, 'Challenged': 1, 'Critical': 2}
ID2LABEL   = {0: 'Stable', 1: 'Challenged', 2: 'Critical'}

print('\n✓ All imports successful')

Working directory: d:\Programming\Projects\Mental Health Assessment

✓ All imports successful


## Cell 2 — Load Dataset · Encode Labels · Stratified Split · Helpers

Loads `mha_text_dataset.csv` (produced by Notebook 01 with columns `Student Information` and `Mental Health Status`). The text column is renamed to `text` and the label column is encoded to integers (`Stable=0`, `Challenged=1`, `Critical=2`) and renamed to `label` — both required by the HuggingFace Trainer.

A stratified 80/20 split is applied and both splits are converted to HuggingFace `Dataset` objects.

`compute_metrics(eval_pred)` is the function passed to the Trainer. It computes macro F1 (returned as `f1` so the Trainer can use it for checkpoint selection via `metric_for_best_model='f1'`), weighted F1, and accuracy.

`fine_tune_transformer(model_name, model_id, train_ds, test_ds)` is a reusable helper that handles: tokenisation, model loading, TrainingArguments, Trainer setup, training, history parsing, results saving, plot generation, checkpoint cleanup, and metric reporting. Each model cell (3–7) calls it with just the model name and HuggingFace ID.

In [2]:
# ── Load and prepare dataset ──────────────────────────────────────────────────
df_text = pd.read_csv(os.path.join('data', 'processed', 'mha_text_dataset.csv'))
df_text = df_text.rename(columns={
    'Student Information' : 'text',
    'Mental Health Status': 'label_str',
})
df_text['label'] = df_text['label_str'].map(LABEL_MAP)

print(f'Loaded : {df_text.shape}')
print(f'Columns: {list(df_text.columns)}')
print(f'Label distribution:')
print(df_text['label_str'].value_counts().to_string())

# Stratified 80/20 split
train_df, test_df = train_test_split(
    df_text[['text', 'label']],
    test_size=0.2, random_state=SEED,
    stratify=df_text['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

print(f'\nTrain : {len(train_dataset)} samples')
print(f'Test  : {len(test_dataset)} samples')


# ── Metrics function ──────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    """Returns macro F1 (key: 'f1'), weighted F1, and accuracy."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1'         : f1_score(labels, preds, average='macro',    zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
        'accuracy'   : accuracy_score(labels, preds),
    }


# ── Fine-tuning helper ────────────────────────────────────────────────────────
def fine_tune_transformer(model_name, model_id, train_ds, test_ds,
                           tokenizer_id=None):
    """
    Fine-tune a HuggingFace transformer for 3-class MHS classification.
    Saves per-epoch results CSV, final metrics CSV, and training plot.
    Returns (final_metrics dict, history DataFrame).
    """
    tok_id = tokenizer_id or model_id
    print(f'  Loading tokenizer : {tok_id}')
    tokenizer = AutoTokenizer.from_pretrained(tok_id)

    def tokenize_fn(examples):
        return tokenizer(examples['text'], truncation=True,
                         padding='max_length', max_length=MAX_LENGTH)

    train_tok = train_ds.map(tokenize_fn, batched=True, batch_size=64)
    test_tok  = test_ds.map(tokenize_fn,  batched=True, batch_size=64)
    train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_tok.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

    print(f'  Loading model     : {model_id}')
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL_MAP,
        ignore_mismatched_sizes=True,
    )

    ckpt_dir = os.path.join('results', 'Natural Language Processing',
                             model_name, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = 5,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 8,
        learning_rate               = 2e-5,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 1,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        greater_is_better           = True,
        use_cpu                     = True,
        dataloader_num_workers      = 0,
        logging_steps               = 50,

        report_to                   = 'none',
        seed                        = SEED,
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_tok,
        eval_dataset    = test_tok,
        compute_metrics = compute_metrics,
    )

    print(f'  Training...')
    trainer.train()

    # Parse per-epoch eval history
    epoch_logs = [log for log in trainer.state.log_history if 'eval_loss' in log]
    history_df = pd.DataFrame(epoch_logs).rename(columns={
        'epoch'          : 'Epoch',
        'eval_loss'      : 'Eval_Loss',
        'eval_f1'        : 'F1_Macro',
        'eval_f1_weighted': 'F1_Weighted',
        'eval_accuracy'  : 'Accuracy',
    })
    keep_cols = [c for c in ['Epoch','Eval_Loss','F1_Macro','F1_Weighted','Accuracy']
                 if c in history_df.columns]
    history_df = history_df[keep_cols]

    # Final evaluation on test set
    final_eval = trainer.evaluate()
    final_metrics = {
        'Model'      : model_name,
        'F1_Macro'   : round(final_eval.get('eval_f1',           0), 4),
        'F1_Weighted': round(final_eval.get('eval_f1_weighted',  0), 4),
        'Accuracy'   : round(final_eval.get('eval_accuracy',     0), 4),
        'Eval_Loss'  : round(final_eval.get('eval_loss',         0), 4),
    }

    # Save results
    res_dir = os.path.join('results', 'Natural Language Processing', model_name)
    os.makedirs(res_dir, exist_ok=True)
    history_df.to_csv(os.path.join(res_dir, f'{model_name}_results.csv'), index=False)
    pd.DataFrame([final_metrics]).to_csv(
        os.path.join(res_dir, f'{model_name}_final_metrics.csv'), index=False
    )

    # Training plot — loss (left) and validation macro F1 (right)
    fig_dir = os.path.join('figures', 'Natural Language Processing', model_name)
    os.makedirs(fig_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df['Epoch'], history_df['Eval_Loss'],
                 marker='o', color='steelblue', linewidth=1.5)
    axes[0].set_title(f'{model_name.upper()} — Validation Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CrossEntropy Loss')

    if 'F1_Macro' in history_df.columns:
        best_f1  = history_df['F1_Macro'].max()
        axes[1].plot(history_df['Epoch'], history_df['F1_Macro'],
                     marker='o', color='green', linewidth=1.5, label='Val Macro F1')
        axes[1].axhline(y=best_f1, color='red', linestyle='--', linewidth=0.8,
                        label=f'Best: {best_f1:.4f}')
        axes[1].set_ylim(0, 1); axes[1].legend()
    axes[1].set_title(f'{model_name.upper()} — Validation Macro F1')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')

    plt.suptitle(f'{model_name.upper()} Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'training_history.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # Save final best model and tokenizer
    model_save_dir = os.path.join('models', 'Natural Language Processing', model_name)
    os.makedirs(model_save_dir, exist_ok=True)
    trainer.save_model(model_save_dir)
    tokenizer.save_pretrained(model_save_dir)
    print(f'  Model saved → {model_save_dir}')

    # Remove checkpoints to free disk space
    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f'  ✓ Done — F1_Macro={final_metrics["F1_Macro"]}  '
          f'Acc={final_metrics["Accuracy"]}')
    return final_metrics, history_df


all_nlp_results = []  # Accumulates final_metrics dicts from Cells 3-7
print('\n✓ Dataset loaded and helpers defined — ready for fine-tuning')

Loaded : (2022, 3)
Columns: ['text', 'label_str', 'label']
Label distribution:
label_str
Critical      1292
Challenged     607
Stable         123

Train : 1617 samples
Test  : 405 samples

✓ Dataset loaded and helpers defined — ready for fine-tuning


## Cell 3 — Fine-Tune BERT (`bert-base-uncased`)

The original BERT base model, pre-trained on BooksCorpus and English Wikipedia (16 GB). Serves as the general-purpose transformer baseline for this task.

Calls `fine_tune_transformer` with `model_name='bert'` and `model_id='bert-base-uncased'`. Saves `bert_results.csv`, `bert_final_metrics.csv`, and `training_history.png`. Appends final metrics to `all_nlp_results`.

In [3]:
print('============================================================')
print('Fine-tuning: BERT')
print('============================================================')

metrics_bert, history_bert = fine_tune_transformer(
    model_name = 'bert',
    model_id   = 'bert-base-uncased',
    train_ds   = train_dataset,
    test_ds    = test_dataset,
)
all_nlp_results.append(metrics_bert)

print(f'History:')
print(history_bert.to_string(index=False))

Fine-tuning: BERT
  Loading tokenizer : bert-base-uncased


Map:   0%|          | 0/1617 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

  Loading model     : bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,F1,F1 Weighted,Accuracy
0.604802,0.698590,5,0.601390,0.710711,0.711111


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved → models\Natural Language Processing\bert
  ✓ Done — F1_Macro=0.6014  Acc=0.7111
History:
 Epoch  Eval_Loss  F1_Macro  F1_Weighted  Accuracy
   1.0   0.658132  0.546773     0.690706  0.691358
   2.0   0.698966  0.601594     0.710359  0.711111
   3.0   0.711027  0.551020     0.689107  0.723457
   4.0   0.631032  0.578736     0.693255  0.688889
   5.0   0.634908  0.588634     0.692119  0.683951


## Cell 4 — Fine-Tune DistilBERT (`distilbert-base-uncased`)

A distilled version of BERT that retains 97% of its language understanding while being 40% smaller and 60% faster. Particularly advantageous on CPU given the runtime constraints.

Calls `fine_tune_transformer` with `model_name='distilbert'` and `model_id='distilbert-base-uncased'`. Saves `distilbert_results.csv`, `distilbert_final_metrics.csv`, and `training_history.png`. Appends final metrics to `all_nlp_results`.

In [4]:
print('============================================================')
print('Fine-tuning: DistilBERT')
print('============================================================')

metrics_distilbert, history_distilbert = fine_tune_transformer(
    model_name = 'distilbert',
    model_id   = 'distilbert-base-uncased',
    train_ds   = train_dataset,
    test_ds    = test_dataset,
)
all_nlp_results.append(metrics_distilbert)

print(f'History:')
print(history_distilbert.to_string(index=False))

Fine-tuning: DistilBERT
  Loading tokenizer : distilbert-base-uncased


Map:   0%|          | 0/1617 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

  Loading model     : distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,F1 Weighted,Accuracy
0.596391,0.646267,5,0.621582,0.705399,0.698765


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved → models\Natural Language Processing\distilbert
  ✓ Done — F1_Macro=0.6216  Acc=0.6988
History:
 Epoch  Eval_Loss  F1_Macro  F1_Weighted  Accuracy
   1.0   0.659577  0.589768     0.689343  0.681481
   2.0   0.673052  0.587327     0.701766  0.701235
   3.0   0.694301  0.586514     0.698848  0.713580
   4.0   0.641501  0.610167     0.707944  0.706173
   5.0   0.646267  0.621582     0.705399  0.698765


## Cell 5 — Fine-Tune ALBERT (`albert-base-v2`)

A Lite BERT with two parameter-reduction techniques: factorised embedding parameterisation and cross-layer parameter sharing. Substantially fewer parameters than BERT base with competitive performance.

Calls `fine_tune_transformer` with `model_name='albert'` and `model_id='albert-base-v2'`. Saves `albert_results.csv`, `albert_final_metrics.csv`, and `training_history.png`. Appends final metrics to `all_nlp_results`.

In [5]:
print('============================================================')
print('Fine-tuning: ALBERT')
print('============================================================')

metrics_albert, history_albert = fine_tune_transformer(
    model_name = 'albert',
    model_id   = 'albert-base-v2',
    train_ds   = train_dataset,
    test_ds    = test_dataset,
)
all_nlp_results.append(metrics_albert)

print(f'History:')
print(history_albert.to_string(index=False))

Fine-tuning: ALBERT
  Loading tokenizer : albert-base-v2


Map:   0%|          | 0/1617 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

  Loading model     : albert-base-v2


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,F1 Weighted,Accuracy
0.641747,0.692131,5,0.461074,0.695122,0.720988


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved → models\Natural Language Processing\albert
  ✓ Done — F1_Macro=0.4611  Acc=0.721
History:
 Epoch  Eval_Loss  F1_Macro  F1_Weighted  Accuracy
   1.0   0.731992  0.452850     0.677685  0.693827
   2.0   0.692131  0.461074     0.695122  0.720988
   3.0   0.754628  0.287219     0.524823  0.646914
   4.0   0.752278  0.451242     0.679857  0.701235
   5.0   0.665023  0.459388     0.687741  0.706173


## Cell 6 — Fine-Tune BioBERT (`dmis-lab/biobert-v1.1`)

BERT further pre-trained on PubMed abstracts (4.5 billion words) and PubMed Central full-text articles (13.5 billion words). Expected to capture biomedical terminology used in PSS, GAD, and PHQ item descriptions.

Calls `fine_tune_transformer` with `model_name='biobert'` and `model_id='dmis-lab/biobert-v1.1'`. Saves `biobert_results.csv`, `biobert_final_metrics.csv`, and `training_history.png`. Appends final metrics to `all_nlp_results`.

In [6]:
print('============================================================')
print('Fine-tuning: BioBERT')
print('============================================================')

metrics_biobert, history_biobert = fine_tune_transformer(
    model_name = 'biobert',
    model_id   = 'dmis-lab/biobert-v1.1',
    train_ds   = train_dataset,
    test_ds    = test_dataset,
)
all_nlp_results.append(metrics_biobert)

print(f'History:')
print(history_biobert.to_string(index=False))

Fine-tuning: BioBERT
  Loading tokenizer : dmis-lab/biobert-v1.1


Map:   0%|          | 0/1617 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

  Loading model     : dmis-lab/biobert-v1.1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,F1 Weighted,Accuracy
0.605221,0.711222,5,0.614909,0.716883,0.720988


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved → models\Natural Language Processing\biobert
  ✓ Done — F1_Macro=0.6149  Acc=0.721
History:
 Epoch  Eval_Loss  F1_Macro  F1_Weighted  Accuracy
   1.0   0.661871  0.465670     0.689416  0.701235
   2.0   0.721072  0.573501     0.685294  0.686420
   3.0   0.711222  0.614909     0.716883  0.720988
   4.0   0.645238  0.584558     0.697901  0.691358
   5.0   0.636509  0.603932     0.699869  0.691358


## Cell 7 — Fine-Tune ClinicalBERT (`emilyalsentzer/Bio_ClinicalBERT`)

BioBERT further pre-trained on MIMIC-III clinical notes (880 million words). Clinical notes share vocabulary with mental health assessment language — expected to generalise well to this domain.

Calls `fine_tune_transformer` with `model_name='clinicalbert'` and `model_id='emilyalsentzer/Bio_ClinicalBERT'`. Saves `clinicalbert_results.csv`, `clinicalbert_final_metrics.csv`, and `training_history.png`. Appends final metrics to `all_nlp_results`.

In [7]:
print('============================================================')
print('Fine-tuning: ClinicalBERT')
print('============================================================')

metrics_clinicalbert, history_clinicalbert = fine_tune_transformer(
    model_name = 'clinicalbert',
    model_id   = 'emilyalsentzer/Bio_ClinicalBERT',
    train_ds   = train_dataset,
    test_ds    = test_dataset,
)
all_nlp_results.append(metrics_clinicalbert)

print(f'History:')
print(history_clinicalbert.to_string(index=False))

Fine-tuning: ClinicalBERT
  Loading tokenizer : emilyalsentzer/Bio_ClinicalBERT


Map:   0%|          | 0/1617 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

  Loading model     : emilyalsentzer/Bio_ClinicalBERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,F1 Weighted,Accuracy
0.567865,0.603684,5,0.638394,0.738391,0.738272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model saved → models\Natural Language Processing\clinicalbert
  ✓ Done — F1_Macro=0.6384  Acc=0.7383
History:
 Epoch  Eval_Loss  F1_Macro  F1_Weighted  Accuracy
   1.0   0.626781  0.528220     0.686170  0.686420
   2.0   0.612255  0.619081     0.729990  0.730864
   3.0   0.603684  0.638394     0.738391  0.738272
   4.0   0.612531  0.629001     0.721677  0.716049
   5.0   0.615551  0.621064     0.706155  0.696296


## Cell 8 — Save NLP Summary CSV

Consolidates the final metrics from all five models into a single DataFrame and saves to `summary/Results/Natural Language Processing/nlp_results_summary.csv`. Results are sorted by **F1_Macro** — the primary metric consistent with all previous notebooks. A completion message lists all outputs produced.

In [8]:
nlp_summary = pd.DataFrame(all_nlp_results)

os.makedirs(os.path.join('summary', 'Results', 'Natural Language Processing'), exist_ok=True)
NLP_SUMMARY = os.path.join('summary', 'Results', 'Natural Language Processing',
                            'nlp_results_summary.csv')
nlp_summary.to_csv(NLP_SUMMARY, index=False)

print(f'✓ Saved : {NLP_SUMMARY}')
print(f'  Shape : {nlp_summary.shape}  (expected 5 rows)')
print()
print('NLP Results sorted by F1_Macro (primary metric):')
print(
    nlp_summary
    .sort_values('F1_Macro', ascending=False)
    .to_string(index=False)
)
print()
print('── Notebook 05 complete ──')
print('  models/Natural Language Processing/    5 saved model folders')
print('  results/Natural Language Processing/  10 CSVs (5 results + 5 final_metrics)')
print('  figures/Natural Language Processing/   5 training_history.png files')
print('  summary/Results/Natural Language Processing/nlp_results_summary.csv')

✓ Saved : summary\Results\Natural Language Processing\nlp_results_summary.csv
  Shape : (5, 5)  (expected 5 rows)

NLP Results sorted by F1_Macro (primary metric):
       Model  F1_Macro  F1_Weighted  Accuracy  Eval_Loss
clinicalbert    0.6384       0.7384    0.7383     0.6037
  distilbert    0.6216       0.7054    0.6988     0.6463
     biobert    0.6149       0.7169    0.7210     0.7112
        bert    0.6014       0.7107    0.7111     0.6986
      albert    0.4611       0.6951    0.7210     0.6921

── Notebook 05 complete ──
  models/Natural Language Processing/    5 saved model folders
  results/Natural Language Processing/  10 CSVs (5 results + 5 final_metrics)
  figures/Natural Language Processing/   5 training_history.png files
  summary/Results/Natural Language Processing/nlp_results_summary.csv
